# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1 — Distribution observations

I inspected six candidate signals to understand their distributions before testing their usefulness.

- `impressions_90d` is strongly right-skewed. The median is 731, while the mean is 5,200.37 and the 99th percentile is 73,505.83, showing that a small number of pages have very high impressions.
- `clicks_90d` is also strongly right-skewed. The median is only 1 click, compared with a mean of 16.10 and a 99th percentile of 253.01.
- `content_age_days` has a median of 236 days and a mean of 256.17 days, with a 99th percentile of 537 days.
- `days_since_last_update` has a median of 20 days and a mean of 46.10 days. Its upper values extend to 373 days.
- `ctr` is highly right-skewed. The median is 0.07, while the mean is 0.51 and the 99th percentile is 8.33.
- `avg_position` is also right-skewed, with a median of 10.80, mean of 16.34, and 99th percentile of 69.90.

There are 1,205 rows where `avg_position == 0`. According to the dataset definition, these rows represent no available position data rather than a true rank of zero.

Overall, the distributions show substantial skew and heavy tails, especially for impressions and clicks. Therefore, bucket-based comparisons are more appropriate for the upcoming signal tests than relying only on averages.

This section is descriptive only; no signal verdict is made here.

In [11]:
import pandas as pd
from pathlib import Path

# Use the repository's starter dataset; do not alter the data.
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), (
    "Starter CSV not found — run this notebook from the repository root."
)
df = pd.read_csv(DATA_PATH)

fields = [
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

missing_fields = sorted(set(fields) - set(df.columns))
assert not missing_fields, f"Missing expected columns: {missing_fields}"

# n is the non-missing count; missing is shown separately.
summary = pd.DataFrame({
    "n (non-missing)": df[fields].count(),
    "missing": df[fields].isna().sum(),
    "min": df[fields].min(),
    "median": df[fields].median(),
    "mean": df[fields].mean(),
    "max": df[fields].max(),
})
summary.index.name = "field"

quantiles = df[fields].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
quantiles.columns = ["p25", "p50", "p75", "p90", "p95", "p99"]
quantiles.index.name = "field"

print("Distribution summary")
print(summary.round(2).to_string())
print("\nQuantiles (use the upper percentiles to spot long or heavy tails)")
print(quantiles.round(2).to_string())

zero_position_rows = (df["avg_position"] == 0).sum()
print(
    f"avg_position == 0: {zero_position_rows:,} rows "
    "(0 means no position data, not rank zero)."
)


Distribution summary
                        n (non-missing)  missing   min  median     mean       max
field                                                                            
impressions_90d                   30000        0   1.0  731.00  5200.37  517715.0
clicks_90d                        30000        0   0.0    1.00    16.10    4178.0
content_age_days                  30000        0  90.0  236.00   256.17     564.0
days_since_last_update            30000        0   1.0   20.00    46.10     373.0
ctr                               30000        0   0.0    0.07     0.51     100.0
avg_position                      30000        0   0.0   10.80    16.34     245.0

Quantiles (use the upper percentiles to spot long or heavy tails)
                          p25     p50      p75       p90       p95       p99
field                                                                       
impressions_90d          81.0  731.00  3615.25  12136.40  22996.50  73505.83
clicks_90d               

## 2. Signal test #1 / #2 / #3 (verdict each)

I tested three signals to see whether pages in different groups had different observed decline rates. I used bucket-based comparisons so that I could see how the decline rate changes across different ranges of each signal.

### Signal test 1 — Staleness / refresh: `days_since_last_update`

This shows how many days it has been since a page was last updated. I expected that pages which had not been updated for a longer time might have a higher decline rate.

The decline rate increased from **51.1%** for pages updated within 30 days to **61.1%** for pages that had not been updated for 91–180 days. However, it dropped to **47.1%** for the 181+ day group. The 31–90 day and 181+ day groups are also small, with fewer than 200 pages each.

**Verdict: MIXED**

The results show some support for the staleness idea, but the decline rate does not continue increasing for every bucket.

### Signal test 2 — Volume / quick win: `impressions_90d`

This shows how many search impressions a page received during the 90-day period. I checked whether pages with different levels of search visibility had different decline rates.

The decline rate was **45.4%** for pages with 1–299 impressions. It increased to **61.5%** for 300–2,999 impressions and was **58.6%** for 3,000–29,999 impressions. However, it dropped to **46.2%** for pages with 30,000+ impressions.

**Verdict: MIXED**

The middle-volume groups have higher decline rates, but there is no consistent increase or decrease across all volume ranges.

### Signal test 3 — Search ranking: `avg_position`

This shows the average search position of a page, where a lower positive value represents a better ranking. I checked whether decline rates changed across different ranking ranges.

For pages with position data, the decline rate was **49.8%** for positions 1–3, **56.9%** for positions 4–10, and reached **61.0%** for positions 11–20. It then decreased to **56.2%** for positions 21–50 and **34.3%** for positions 51+.

The `avg_position == 0` group represents **no position data**, so I did not treat it as an actual ranking position.

**Verdict: MIXED**

The decline rate changes across ranking groups, but there is no simple pattern where worse ranking always means a higher decline rate.

### Overall observation

All three signals show differences in decline rates between their buckets, but none of them has a simple and consistent relationship across the whole range. This means the signals may still be useful for understanding pages, but they should not be used alone to decide which pages need action.

In [12]:
# The starter-data label is used only to calculate descriptive decline rates.
# It is not a signal feature.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

def print_bucket_table(frame, bucket_column, title, bucket_order):
    table = (
        frame.groupby(bucket_column, observed=False)["is_declining_label"]
        .agg(n="size", decline_count="sum", decline_rate="mean")
        .reindex(bucket_order)
        .reset_index()
        .rename(columns={bucket_column: "bucket"})
    )
    table["decline_rate"] = (table["decline_rate"] * 100).round(1)
    print(f"\n{title}")
    print(table.to_string(index=False))

# 1. Staleness / refresh: readable update-age ranges.
staleness_order = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=staleness_order,
)
print_bucket_table(
    df, "staleness_bucket", "1. Staleness / refresh signal", staleness_order
)

# 2. Volume / quick win: the traffic bands are intentionally broad for a heavy-tailed metric.
volume_order = [
    "1-299 impressions", "300-2,999 impressions",
    "3,000-29,999 impressions", "30,000+ impressions",
]
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 299, 2_999, 29_999, float("inf")],
    labels=volume_order,
    include_lowest=True,
)
print_bucket_table(df, "volume_bucket", "2. Volume / quick-win signal", volume_order)

# 3. Search ranking: zero is a distinct no-position-data state, never rank zero.
position_order = [
    "no position data (0)", "top 3 (1-3)", "page 1 (4-10)",
    "striking distance (11-20)", "pages 3-5 (21-50)", "deep (51+)",
]
position_with_data = pd.cut(
    df["avg_position"].where(df["avg_position"].ne(0)),
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=position_order[1:],
    include_lowest=True,
)
df["position_bucket"] = pd.Categorical(
    position_with_data.astype(object).where(
        df["avg_position"].ne(0), "no position data (0)"
    ),
    categories=position_order,
    ordered=True,
)
print_bucket_table(df, "position_bucket", "3. Search-ranking signal", position_order)

print("\nReview the observed tables, then write one permitted verdict in the markdown cell above.")



1. Staleness / refresh signal
     bucket     n  decline_count  decline_rate
  0-30 days 20480          10473          51.1
 31-90 days   175            103          58.9
91-180 days  9171           5604          61.1
  181+ days   174             82          47.1

2. Volume / quick-win signal
                  bucket     n  decline_count  decline_rate
       1-299 impressions 11248           5106          45.4
   300-2,999 impressions 10469           6435          61.5
3,000-29,999 impressions  7205           4223          58.6
     30,000+ impressions  1078            498          46.2

3. Search-ranking signal
                   bucket     n  decline_count  decline_rate
     no position data (0)  1205              8           0.7
              top 3 (1-3)  1141            568          49.8
            page 1 (4-10) 11842           6743          56.9
striking distance (11-20)  7273           4433          61.0
        pages 3-5 (21-50)  7225           4059          56.2
            

## 3. The flag-linked test

### Staleness / refresh flag assumption

I tested the assumption behind FlyRank's staleness / refresh flag: pages that have not been updated for a longer time may be more likely to decline.

For this test, I divided the pages into two groups:

- **Not stale:** fewer than 91 days since the last update
- **Stale:** 91 days or more since the last update

### What did the data show?

The **Not stale** group had an observed decline rate of **51.20%**, while the **Stale** group had a decline rate of **60.85%**.

This means the stale group had a **9.65 percentage-point higher** observed decline rate than the not-stale group.

### What does this tell us?

The result provides **descriptive support** for the staleness / refresh assumption because the stale pages had a higher observed decline rate.

However, this does not mean that being stale causes a page to decline. It only shows that the two groups had different observed decline rates in this dataset.

### Human-review limitation

A page may not have been updated recently because the content is already accurate, evergreen, or performing well. Therefore, staleness should be treated as a signal for **human review**, rather than an automatic reason to refresh a page.

In [13]:
# Use the documented starter-data label only for this descriptive comparison.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

staleness_flag = df["days_since_last_update"].ge(91)
flag_table = (
    df.assign(
        stale_status=staleness_flag.map(
            {False: "Not stale (<91 days)", True: "Stale (91+ days)"}
        )
    )
    .groupby("stale_status")["is_declining_label"]
    .agg(n="size", decline_count="sum", decline_rate="mean")
    .reindex(["Not stale (<91 days)", "Stale (91+ days)"])
    .reset_index()
)
flag_table["decline_rate"] = (flag_table["decline_rate"] * 100).round(2)

not_stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Not stale (<91 days)"), "decline_rate"
].iloc[0]
stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Stale (91+ days)"), "decline_rate"
].iloc[0]
rate_difference_pp = stale_rate - not_stale_rate

print("Staleness / refresh flag comparison")
print(flag_table.to_string(index=False))
print(f"\nStale minus not-stale decline rate: {rate_difference_pp:+.2f} percentage points")


Staleness / refresh flag comparison
        stale_status     n  decline_count  decline_rate
Not stale (<91 days) 20655          10576         51.20
    Stale (91+ days)  9345           5686         60.85

Stale minus not-stale decline rate: +9.65 percentage points


## 4. What this means in practice

From these tests, I found that none of the three signals gives a clear and consistent pattern across all buckets. So I would not use any single signal by itself to decide that a page needs to be refreshed.

The staleness signal was interesting. In the flag-linked test, pages that had not been updated for 91 days or more had a **60.85%** decline rate, compared with **51.20%** for pages updated more recently. This gives some support to the idea behind the refresh flag.

However, this does not mean that old content causes a decline. There can be other reasons why a page has not been updated, such as the content already being accurate or evergreen.

The volume and search-position tests also showed mixed patterns. Because of this, I think these signals are better used together as supporting information for **human review**, rather than as automatic decisions.

The main takeaway is that a signal can show some useful pattern without being strong enough to become a decision rule on its own. This is something I will keep in mind when building the baseline action score in ML-07.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.